In [2]:
import pandas as pd
import scraping_functions as sf
import importlib

importlib.reload(sf);

In [3]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## Saving Data into CSV

Collecting data from Transfermarkt requires sending a large number of requests across multiple seasons and rounds, which can be time-consuming. To avoid repeating the scraping process every time the data is needed, the extracted information is saved as CSV files. For this project, the historical database covers all Premier League seasons from 1992 to 2025 (the latest completed season at the time of development).

### Iterating Through Functions That Only Require Seasons (Faster)

Some scraping functions only require the season as input, such as the final league table, champions, and squad information. These functions can be executed with a single loop that iterates through all seasons and stores the results in CSV files.

In [9]:
premier_table = []
premier_squad = []
premier_top_scorers = []
league = 'premier-league'


for season in range(1992, 2026):
    # Creating the table list
    table_season_data = sf.get_table(headers, league, season)
    premier_table.extend(table_season_data[1:])

    # Creating the squad List
    squad_season_data = sf.get_squad(headers,league,season)
    premier_squad.extend(squad_season_data[1:])

    # Creating the top scorers List
    table_data = sf.get_top_scorers(headers,league,season)
    premier_top_scorers.extend(table_data[1:])

# Transforming to a pandas Data Frame
df_premier_table = pd.DataFrame(premier_table, columns=table_season_data[0])
df_premier_squad = pd.DataFrame(premier_squad, columns=squad_season_data[0])
df_premier_top_scorers = pd.DataFrame(premier_top_scorers,columns=table_data[0])
df_premier_top_scorers.sort_values(['season_id','pos'],inplace=True,ignore_index=True)

# Exporting to csv
df_premier_table.to_csv('../data-csv/premier-table.csv', index=False)
df_premier_squad.to_csv('../data-csv/premier-squad.csv', index=False)
df_premier_top_scorers.to_csv('../data-csv/premier-top-scorers.csv', index=False)

### Iterating Through Functions That Require Seasons and Rounds

Other functions, such as match results, match events, and round-by-round standings, require both the season and the round as input. These functions are executed using nested loops, iterating through every round of every season before saving the collected data.

In [6]:
premier_events = []
premier_matches = []
premier_placements = []
league = 'premier-league'

for season in range(2024,2026):
    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        # Creating the events List
        event_season_data = sf.get_events(headers,league,season,n_round)
        premier_events.extend(event_season_data[1:])

        # Creating the matches List
        matches_data = sf.get_match(headers,league,season,n_round)
        premier_matches.extend(matches_data[1:])

        # Creating the placements List
        placements_data = sf.get_placements(headers,league,season,n_round)
        premier_placements.extend(placements_data[1:])

df_premier_events = pd.DataFrame(premier_events,columns=event_season_data[0])
df_premier_matches = pd.DataFrame(premier_matches, columns=matches_data[0])
df_premier_placements = pd.DataFrame(premier_placements, columns=placements_data[0])

df_premier_events.to_csv("../data-csv/premier-events.csv", index=False)
df_premier_matches.to_csv("../data-csv/premier-matches.csv", index=False)
df_premier_placements.to_csv("../data-csv/premier-placements.csv", index=False)


### Updating the Database with Future Seasons

Once the initial historical database has been created, future seasons can be added without rebuilding the entire dataset. The same functions can simply be executed for the new season, and the results appended to the existing CSV files, making the project scalable, reusable, and easy to maintain.

In [10]:
event_extended = pd.read_csv('../data-csv/premier-events.csv', encoding='UTF-8')
league = 'premier-league'

for season in range(2022,2026):
    season_check = event_extended['season_id'].str.contains(str(season), case=False).any()

    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        round_check = event_extended['match_id'].str.contains(str(n_round), case=False).any()
        if round_check and season_check:
            break
        else:
            data = sf.get_events(headers,league,season,n_round)
            temp = pd.DataFrame(data[1:],columns=data[0])
            event_extended = pd.concat([event_extended,temp])    

event_extended.sort_values(['event_id'],ignore_index=True,inplace=True)
display(event_extended)

,season_id,match_id,event_id,event_team,event_minute,event_type,event_player
0,GB1-2022,M-2022-01-01,E-2022-01-0001,Arsenal FC,20',1,Gabriel Martinelli
1,GB1-2022,M-2022-01-01,E-2022-01-0002,Arsenal FC,85',3,Marc Guéhi
2,GB1-2022,M-2022-01-02,E-2022-01-0003,Fulham FC,32',1,Aleksandar Mitrović
3,GB1-2022,M-2022-01-02,E-2022-01-0004,Liverpool FC,64',1,Darwin Núñez
4,GB1-2022,M-2022-01-02,E-2022-01-0005,Fulham FC,72',2,Aleksandar Mitrović
...,...,...,...,...,...,...,...
4734,GB1-2025,M-2025-38-08,E-2025-38-0021,Chelsea FC,62',-3,Wesley Fofana
4735,GB1-2025,M-2025-38-09,E-2025-38-0022,Tottenham Hotspur,43',1,João Palhinha
4736,GB1-2025,M-2025-38-10,E-2025-38-0023,West Ham United,67',1,Taty Castellanos
4737,GB1-2025,M-2025-38-10,E-2025-38-0024,West Ham United,79',1,Jarrod Bowen
